# LeetCode #183 — Customers Who Never Order (Khách hàng chưa từng đặt hàng)

**Vị trí 43 trong danh sách Easy — giáo trình SQL nhập môn, thực hành thật bằng Python + SQLite.**  
Đề gốc: https://leetcode.com/problems/customers-who-never-order/

> **Chú ý format:** Bài Database cần nộp **câu SQL**. Python trong notebook chỉ dựng database, chạy các câu SQL và kiểm thử. Trên LeetCode, không dán `import sqlite3` hay `class Solution`.

**Mục tiêu học:**
1. Hiểu hai bảng `Customers` và `Orders`, khóa chính và khóa ngoại `customerId`.
2. Tự dùng `SELECT`, `WHERE`, `JOIN`, `LEFT JOIN`, `ON`, `IS NULL` từ các ví dụ nhỏ.
3. Hiểu vì sao muốn tìm **không có đơn hàng** thì phải **giữ khách hàng chưa ghép được**.
4. Trace bằng tay và bằng code: với mỗi khách hàng, có đơn khớp nào không? Kết quả ở đâu?
5. Viết được hai lời giải chuẩn `LEFT JOIN ... IS NULL` và `NOT EXISTS`.
6. Tránh các lỗi `INNER JOIN`, `NOT IN` với `NULL`, thiếu điều kiện liên kết; hiểu complexity phụ thuộc query plan.
7. Làm 6 bài tập tăng dần, có 3 cấp gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown trước, đoán bảng kết quả, chạy cell để kiểm tra; không cần cài MySQL hay pandas.

## 1. Đề bài bằng lời thường — ai chưa từng mua hàng?

Cửa hàng có hai bảng:

**Customers** (danh sách tất cả khách hàng):

| id | name |
|---:|---|
| 1 | Joe |
| 2 | Henry |
| 3 | Sam |
| 4 | Max |

**Orders** (mỗi dòng là một đơn hàng):

| id | customerId |
|---:|---:|
| 1 | 3 |
| 2 | 1 |

- Đơn hàng **1** có `customerId = 3` → **Sam** đã đặt.
- Đơn hàng **2** có `customerId = 1` → **Joe** đã đặt.
- Không đơn nào có `customerId = 2` hoặc `4` → **Henry**, **Max** chưa từng đặt.

**Đầu ra yêu cầu:** đúng **một cột `Customers`** (viết hoa C) với tên `Henry` và `Max`, thứ tự tùy ý. **Không được trả ID thay cho tên.**

In [1]:
customers183 = [(1,'Joe'),(2,'Henry'),(3,'Sam'),(4,'Max')]
orders183 = [(1,3),(2,1)]
for customer_id, name in customers183:
    has_order = any(cid == customer_id for _, cid in orders183)
    print(f'{name:<8} | có đơn? {has_order} | cần lấy? {not has_order}')

Joe      | có đơn? True | cần lấy? False
Henry    | có đơn? False | cần lấy? True
Sam      | có đơn? True | cần lấy? False
Max      | có đơn? False | cần lấy? True


## 2. Đọc bảng SQL: `table`, `row`, `column`, `primary key`

- **Table** là bảng, chẳng hạn `Customers`.
- **Row** là bản ghi, chẳng hạn `(2, 'Henry')`.
- **Column** là một thuộc tính, như `name` hoặc `customerId`.
- **Primary key** (khóa chính): `Customers.id` và `Orders.id` riêng biệt; mỗi cột `id` duy nhất **trong bảng của nó**.
- **Foreign key** (khóa ngoại): `Orders.customerId` liên kết tới **`Customers.id`**; cùng giá trị thì đơn hàng thuộc về khách đó.

Đừng nhầm `Orders.id` (mã **đơn hàng**) và `Orders.customerId` (mã **khách đặt**). Trong mẫu có dòng Orders id=1, customerId=3, vậy người đặt là Sam id=3 chứ không phải Joe id=1.

**Liên kết phải dùng:** `Customers.id = Orders.customerId`.

## 3. `NULL` nghĩa là gì? Khác số 0 và chuỗi trống

`NULL` trong SQL biểu thị **chưa có / không biết giá trị** ở vị trí dữ liệu đó. Khi dùng `LEFT JOIN`, nếu một khách hàng **không có đơn** để ghép, các cột lấy từ bảng đơn hàng sẽ là `NULL`.

- `NULL` không phải chuỗi văn bản `'NULL'`.
- `NULL` không phải số `0`.
- Trong Python, SQLite chuyển SQL `NULL` thành `None` khi trả kết quả.
- Trong SQL để kiểm tra thiếu giá trị ta dùng **`IS NULL`**, không dùng `= NULL`.

Ta sẽ thấy `NULL` một cách trực quan trong bảng join, thay vì học định nghĩa một cách máy móc.

## 4. Dùng thư viện `sqlite3` để chạy SQL thật

`sqlite3.connect(':memory:')` tạo cơ sở dữ liệu **trong RAM**, không ghi file. Python chỉ đóng vai trò “người gửi lệnh SQL”.

- `execute` thực thi lệnh SQL.
- `executemany` thêm nhiều dòng, mỗi dấu `?` là một vị trí lấy dữ liệu Python truyền vào.
- `fetchall()` lấy toàn bộ dòng kết quả.

Bài dùng cú pháp cơ bản tương thích MySQL và SQLite. Khi copy lời giải lên LeetCode, bạn chỉ cần **SELECT ... LEFT JOIN ... WHERE ...**.

In [2]:
import sqlite3
db183 = sqlite3.connect(':memory:')
print('SQLite sẵn sàng:',sqlite3.sqlite_version)

SQLite sẵn sàng: 3.46.1


## 5. Tạo schema (cấu trúc) cho cả hai bảng

Câu `CREATE TABLE` dựng cột và kiểu dữ liệu. Để dễ chạy trong SQLite:

```sql
CREATE TABLE Customers (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE Orders (id INTEGER PRIMARY KEY, customerId INTEGER);
```

`PRIMARY KEY` xác định một dòng duy nhất. Trong mô tả gốc, hai ID là `int`, còn `name` là `varchar`; SQLite dùng `INTEGER`/`TEXT` cho phòng lab tương đương về ý nghĩa ở đây.

Ta **không cần cài foreign key constraint** để thử phép `JOIN`; điều kiện nối vẫn được viết tường minh trong câu truy vấn. Các dữ liệu mẫu giữ đúng liên kết khách–đơn.

In [3]:
db183.executescript('''CREATE TABLE Customers (id INTEGER PRIMARY KEY, name TEXT);
CREATE TABLE Orders (id INTEGER PRIMARY KEY, customerId INTEGER);''')
print('Đã tạo Customers và Orders')

Đã tạo Customers và Orders


## 6. Chèn dữ liệu mẫu — đọc kỹ vị trí giá trị

Trong SQL, `INSERT INTO Orders VALUES (1, 3)` có nghĩa **đơn hàng 1** của **khách hàng 3**. Giá trị thứ hai tương ứng cột `customerId`, không phải tên người.

`db183.executemany(...)` chỉ là tiện ích Python để thêm nhiều dòng; dùng `?` giúp giữ riêng câu lệnh SQL và giá trị truyền vào.

**Hãy đoán:** sau khi thêm 4 khách và 2 đơn, nếu liệt kê mọi dòng khách–đơn theo khóa, ta sẽ có mấy cặp khớp? → **2 cặp** (Sam–đơn1, Joe–đơn2).

In [4]:
db183.executemany('INSERT INTO Customers VALUES (?, ?)', customers183)
db183.executemany('INSERT INTO Orders VALUES (?, ?)', orders183)
db183.commit()
print('Đã chèn',len(customers183),'khách và',len(orders183),'đơn')

Đã chèn 4 khách và 2 đơn


## 7. Viết công cụ `show_183` để nhìn tên cột và dữ liệu

Một truy vấn SQL trả về **bảng kết quả**. Python `sqlite3` cho ta hai thứ:

- `cursor.description`: tên các cột đầu ra.
- `cursor.fetchall()`: các dòng đầu ra, mỗi dòng là một tuple.

Hàm `show_183` không thay cho SQL nộp bài. Nó chỉ **in rõ** để người mới nhìn được dữ liệu thay vì chỉ thấy một đối tượng cursor. Trong phòng lab, `ORDER BY` giúp dễ so kết quả; đề chính thức không yêu cầu sắp xếp.

In [5]:
def show_183(query, db=db183):
    cur=db.execute(query)
    columns=[d[0] for d in cur.description]
    rows=cur.fetchall()
    print('Cột:',columns)
    for row in rows:
        print(' ',row)
    print('Số dòng:',len(rows))
    return rows

show_183('SELECT id, name FROM Customers ORDER BY id;')
show_183('SELECT id, customerId FROM Orders ORDER BY id;')

Cột: ['id', 'name']
  (1, 'Joe')
  (2, 'Henry')
  (3, 'Sam')
  (4, 'Max')
Số dòng: 4
Cột: ['id', 'customerId']
  (1, 3)
  (2, 1)
Số dòng: 2


[(1, 3), (2, 1)]

## 8. Học `SELECT` và `FROM` từ ví dụ nhỏ nhất

```sql
SELECT name FROM Customers;
```

- `FROM Customers`: chọn bảng gốc để xem.
- `SELECT name`: chỉ lấy **cột name** từ mỗi dòng.
- Nếu dùng `SELECT *`, ta thấy mọi cột trong bảng, nhưng **đề yêu cầu chỉ đúng một cột tên `Customers`**.

Tên cột có thể đổi ở đầu ra bằng `AS`: `SELECT name AS Customers FROM Customers;`. Việc đặt bí danh không làm thay đổi giá trị trong bảng. Bây giờ ta chưa lọc được ai chưa mua hàng, chỉ mới biết cách lấy tên.

In [6]:
show_183('SELECT name FROM Customers ORDER BY id;')
show_183('SELECT name AS Customers FROM Customers ORDER BY id;')

Cột: ['name']
  ('Joe',)
  ('Henry',)
  ('Sam',)
  ('Max',)
Số dòng: 4
Cột: ['Customers']
  ('Joe',)
  ('Henry',)
  ('Sam',)
  ('Max',)
Số dòng: 4


[('Joe',), ('Henry',), ('Sam',), ('Max',)]

## 9. `WHERE` — lọc từng dòng theo điều kiện

`WHERE id > 2` giữ những khách hàng id=3 hoặc 4. Đây là cách **lọc trên cột của chính bảng đó**.

Nhưng yêu cầu “chưa từng đặt hàng” không thể xác định bằng một cột riêng có sẵn của Customers: khách hàng nào mua hay chưa mua chỉ biết được khi **đối chiếu bảng Orders**.

Do đó chúng ta phải học cách **so sánh hai bảng theo khóa**. Trước hết thử Python thuần để làm rõ tư duy, sau đó mới dùng SQL.

In [7]:
show_183('SELECT name FROM Customers WHERE id > 2 ORDER BY id;')
show_183("SELECT id, name FROM Customers WHERE name = 'Henry';")

Cột: ['name']
  ('Sam',)
  ('Max',)
Số dòng: 2
Cột: ['id', 'name']
  (2, 'Henry')
Số dòng: 1


[(2, 'Henry')]

## 10. Cách dễ nghĩ (baseline): tìm đơn hàng cho từng khách

Với mỗi khách hàng `(id, name)`:

1. Bắt đầu `found = False` (chưa thấy đơn nào).
2. Duyệt **tất cả** dòng của Orders.
3. Nếu `Orders.customerId == Customers.id`, đánh dấu `found = True`.
4. Nếu sau khi tìm vẫn **không tìm thấy**, thêm tên vào kết quả.

Đây là **Brute Force** dùng hai vòng lặp. Nếu có `n` khách và `m` đơn, có thể kiểm tra tới `n × m` cặp → `O(nm)` thời gian. Bài SQL sẽ cho cơ sở dữ liệu tự làm công việc đối chiếu.

In [8]:
def never_order_baseline(customers, orders):
    result=[]
    for customer_id, name in customers:
        found=False
        for _, order_customer_id in orders:
            if customer_id == order_customer_id:
                found=True
                break
        if not found:
            result.append(name)
    return result

print('Ví dụ:',never_order_baseline(customers183,orders183))
assert never_order_baseline(customers183,orders183) == ['Henry','Max']

Ví dụ: ['Henry', 'Max']


## 11. Trace bằng tay: kiểm tra từng khách và từng đơn

Ta ghi rõ cả hành động **khớp** và **không khớp**:

| Khách (`id`) | Đơn có customerId=3 | Đơn có customerId=1 | Có đơn? | Kết luận |
|---|---|---|---|---|
| Joe (1) | không | có | Có | loại |
| Henry (2) | không | không | Không | **chọn** |
| Sam (3) | có | không | Có | loại |
| Max (4) | không | không | Không | **chọn** |

Một khách chỉ cần khớp **một** đơn đã đủ để bị loại. Có 10 đơn cũng không làm khách ấy trở thành “chưa mua”.

Mấu chốt: ta đang tìm những khách **không tồn tại bất kỳ đơn hàng tương ứng**; trong SQL khái niệm này thường gọi là **anti join / anti-existence**.

In [9]:
def trace_183(customers, orders):
    for cid, name in customers:
        matching = [order_id for order_id, customer_id in orders if customer_id == cid]
        print(f'{name:<8} id={cid} | đơn khớp={matching} | chọn? {len(matching)==0}')

trace_183(customers183, orders183)

Joe      id=1 | đơn khớp=[2] | chọn? False
Henry    id=2 | đơn khớp=[] | chọn? True
Sam      id=3 | đơn khớp=[1] | chọn? False
Max      id=4 | đơn khớp=[] | chọn? True


## 12. Hiểu `JOIN ON`: tìm các cặp có khóa đúng

Ta có hai bảng với các cột `id` trùng tên, nên cần **bí danh**:

- `Customers AS c`: gọi ngắn bảng khách là `c`.
- `Orders AS o`: gọi ngắn bảng đơn là `o`.
- `c.id`: ID của **khách**, không phải của đơn.
- `o.customerId`: ID **khách được ghi trên đơn**.

Điều kiện nối đúng là `c.id = o.customerId`.

**`JOIN` (mặc định `INNER JOIN`)** chỉ giữ những cặp có khóa khớp. Hãy tưởng tượng: Henry không có đơn khớp, nên **biến mất khỏi kết quả JOIN thường**. Đây là vấn đề của cách chỉ dùng `JOIN`.

In [10]:
show_183('''SELECT c.id AS customer_id, c.name, o.id AS order_id
FROM Customers c
JOIN Orders o ON c.id = o.customerId
ORDER BY c.id, o.id;''')

Cột: ['customer_id', 'name', 'order_id']
  (1, 'Joe', 2)
  (3, 'Sam', 1)
Số dòng: 2


[(1, 'Joe', 2), (3, 'Sam', 1)]

## 13. `LEFT JOIN`: giữ mọi khách hàng, kể cả khi chưa có đơn

`LEFT JOIN` khác `INNER JOIN` ở một điều cực kỳ quan trọng:

- Lấy **tất cả các dòng bảng bên trái** (ở đây: Customers).
- Nếu có đơn khớp, nối thêm thông tin của Orders.
- Nếu **không có đơn khớp**, vẫn giữ khách; các cột bên phải (Orders) sẽ là `NULL`.

Hãy hình dung kết quả trung gian:

| c.name | o.id |
|---|---:|
| Joe | 2 |
| Henry | NULL |
| Sam | 1 |
| Max | NULL |

**Mấu chốt:** `LEFT JOIN` đã giúp **các khách chưa mua không biến mất**. Ta chỉ còn phải giữ những dòng có **`o.id IS NULL`**.

In [11]:
join183 = '''SELECT c.id AS customer_id, c.name, o.id AS order_id, o.customerId
FROM Customers c
LEFT JOIN Orders o ON c.id = o.customerId
ORDER BY c.id, o.id;'''
show_183(join183)

Cột: ['customer_id', 'name', 'order_id', 'customerId']
  (1, 'Joe', 2, 1)
  (2, 'Henry', None, None)
  (3, 'Sam', 1, 3)
  (4, 'Max', None, None)
Số dòng: 4


[(1, 'Joe', 2, 1),
 (2, 'Henry', None, None),
 (3, 'Sam', 1, 3),
 (4, 'Max', None, None)]

## 14. Vì sao `NULL` chính là dấu hiệu “không có đơn ghép”?

Trong kết quả `LEFT JOIN`, xem cột **`o.id`**:

- Joe: `o.id = 2` → khớp đơn 2, loại.
- Henry: `o.id = NULL` → **không có** đơn nào ghép được, chọn.
- Sam: `o.id = 1` → khớp đơn 1, loại.
- Max: `o.id = NULL` → chưa có đơn, chọn.

Dùng cột `o.id` hợp lý vì đây là **khóa chính** của Orders: nếu có một bản ghi đơn thực sự, `o.id` không NULL. `WHERE o.id IS NULL` vì vậy chọn đúng những dòng khách **không khớp đơn nào**.

Đừng nhầm với trường hợp một đơn hàng **có tồn tại nhưng thông tin ở cột khác trống**: điều đó không có nghĩa khách chưa mua.

In [12]:
show_183('''SELECT c.name, o.id AS order_id
FROM Customers c
LEFT JOIN Orders o ON c.id = o.customerId
WHERE o.id IS NULL
ORDER BY c.id;''')

Cột: ['name', 'order_id']
  ('Henry', None)
  ('Max', None)
Số dòng: 2


[('Henry', None), ('Max', None)]

## 15. Phép kiểm tra `IS NULL` (không phải `= NULL`)

Trong SQL, `NULL` không phải một giá trị thông thường để so sánh bằng. SQL có logic ba trạng thái: True / False / **Unknown**. Khi dùng `x = NULL`, kết quả là Unknown, nên `WHERE` **không giữ dòng**.

Cách đúng để hỏi “cột này đang thiếu giá trị?” là **`x IS NULL`**.

```sql
-- SAI:  WHERE o.id = NULL
-- ĐÚNG: WHERE o.id IS NULL
```

Ta thử một câu `= NULL` hợp cú pháp nhưng sai logic và đối chiếu kết quả.

In [13]:
print('Sai (= NULL):')
show_183('''SELECT c.name FROM Customers c LEFT JOIN Orders o ON c.id=o.customerId WHERE o.id = NULL;''')
print('Đúng (IS NULL):')
show_183('''SELECT c.name FROM Customers c LEFT JOIN Orders o ON c.id=o.customerId WHERE o.id IS NULL;''')

Sai (= NULL):
Cột: ['name']
Số dòng: 0
Đúng (IS NULL):
Cột: ['name']
  ('Henry',)
  ('Max',)
Số dòng: 2


[('Henry',), ('Max',)]

## 16. Câu SQL chính — suy ra từng ý, không học thuộc

Ta ghép bốn ý đã học:

```sql
SELECT c.name AS Customers
FROM Customers AS c
LEFT JOIN Orders AS o
    ON c.id = o.customerId
WHERE o.id IS NULL;
```

**Đọc theo luồng dữ liệu:**
1. `FROM Customers AS c`: bắt đầu từ **mọi khách hàng**.
2. `LEFT JOIN Orders AS o ON c.id = o.customerId`: thử ghép với từng đơn đúng chủ.
3. `WHERE o.id IS NULL`: chỉ giữ các khách **không có đơn khớp**.
4. `SELECT c.name AS Customers`: xuất đúng tên khách vào cột `Customers`.

Trên LeetCode kết quả cho phép bất kỳ thứ tự, nên không cần `ORDER BY`.

In [14]:
sql183 = '''SELECT c.name AS Customers
FROM Customers AS c
LEFT JOIN Orders AS o ON c.id = o.customerId
WHERE o.id IS NULL;'''
actual183 = show_183(sql183)
assert set(actual183) == {('Henry',),('Max',)}
cur=db183.execute(sql183)
assert [col[0] for col in cur.description] == ['Customers']

Cột: ['Customers']
  ('Henry',)
  ('Max',)
Số dòng: 2


## 17. Phiên bản nộp LeetCode chính xác

Chỉ copy **một câu SQL sau** vào khu vực SQL của LeetCode:

```sql
SELECT c.name AS Customers
FROM Customers c
LEFT JOIN Orders o ON c.id = o.customerId
WHERE o.id IS NULL;
```

- `Customers` ở vị trí `FROM` là **tên bảng**.
- `Customers` phía sau `AS` là **tên cột đầu ra** — chúng có thể trùng chữ, không gây sai.
- `o.id IS NULL` kiểm tra **không tồn tại đơn ghép**.
- Không nộp code Python; notebook chỉ là phòng học SQL.

Một câu SQL ngắn vẫn có thể cần nhiều bước tư duy, vì vậy không nên nhảy ngay đến câu này khi học lần đầu.

## 18. Trace tự động mọi bước: trước và sau khi lọc

Ta thực hiện hai truy vấn nối tiếp:

1. Truy vấn **trung gian**: xem toàn bộ bảng trái và cột `order_id` sau LEFT JOIN.
2. Truy vấn **cuối**: chỉ giữ các dòng có `order_id IS NULL`.

Việc theo dõi trạng thái *trước/sau* rất quan trọng vì một lỗi như dùng `INNER JOIN` khiến những người cần tìm đã bị bỏ ngay từ bước 1; **không thể dùng WHERE đưa họ trở lại**.

In [15]:
intermediate = db183.execute(join183).fetchall()
print('Sau LEFT JOIN, trước WHERE:')
for customer_id, name, order_id, order_customer_id in intermediate:
    print(f'  {name:<8} order_id={str(order_id):<5} -> {"GIỮ" if order_id is None else "LOẠI"}')
print('Kết quả cuối:', db183.execute(sql183).fetchall())

Sau LEFT JOIN, trước WHERE:
  Joe      order_id=2     -> LOẠI
  Henry    order_id=None  -> GIỮ
  Sam      order_id=1     -> LOẠI
  Max      order_id=None  -> GIỮ
Kết quả cuối: [('Henry',), ('Max',)]


## 19. Dòng `ON` và dòng `WHERE` trả lời hai câu hỏi khác nhau

`ON c.id = o.customerId` hỏi: “**Đơn này có thuộc khách kia không?**”.

`WHERE o.id IS NULL` hỏi: “**Sau khi đã thử ghép, khách nào không có đơn nào khớp?**”.

Nếu viết nhầm `WHERE c.id = o.customerId`, các khách không ghép được sẽ có `o.customerId = NULL`, điều kiện `=` không True nên **bị lọc mất**. Kết quả đó trở về giống INNER JOIN, đi ngược mục tiêu.

Đối với anti join, cần giữ **điều kiện liên kết trong `ON`** và kiểm tra **không có bản ghi bên phải** trong `WHERE`.

In [16]:
wrong_on_where183 = '''SELECT c.name FROM Customers c
LEFT JOIN Orders o ON 1=1
WHERE c.id=o.customerId ORDER BY c.id;'''
print('Ví dụ sai vì WHERE chỉ giữ khách có đơn:')
show_183(wrong_on_where183)

Ví dụ sai vì WHERE chỉ giữ khách có đơn:
Cột: ['name']
  ('Joe',)
  ('Sam',)
Số dòng: 2


[('Joe',), ('Sam',)]

## 20. Vì sao `INNER JOIN` không giải được bài?

`INNER JOIN` chỉ xuất khách **có đơn**: Joe và Sam. Khi thêm `WHERE o.id IS NULL`, không có dòng ghép thành công nào có `o.id NULL`, nên kết quả rỗng.

Đây là **phản ví dụ** cho lỗi thông dụng: “Cứ cần hai bảng là dùng JOIN”. Trước khi chọn kiểu JOIN, cần hỏi:

- Bài hỏi những dòng **có khớp**? → có thể `INNER JOIN`.
- Bài muốn giữ cả những dòng **không có khớp**? → `LEFT JOIN` hoặc `NOT EXISTS`.

Không có một loại JOIN mặc định đúng cho mọi bài toán.

In [17]:
show_183('''SELECT c.name FROM Customers c INNER JOIN Orders o
ON c.id=o.customerId WHERE o.id IS NULL;''')

Cột: ['name']
Số dòng: 0


[]

## 21. Dựng hàm chạy test trên bảng mới để tránh ảnh hưởng dữ liệu gốc

Muốn thử dữ liệu khác, **không nên xóa rồi chèn lung tung** vào ví dụ ban đầu: các cell sau có thể phụ thuộc vào trạng thái đã thay đổi. Ta tạo database SQLite mới trong RAM cho từng test, chạy query, lấy kết quả, rồi đóng lại.

`run_183(customers, orders, query)` trả về `(tên_cột, các_dòng)`, giúp ta kiểm tra **cả nội dung lẫn format đầu ra**.

Hãy chú ý thứ tự SQL **không được đảm bảo** nếu không ORDER BY; do đó các kiểm thử sẽ so **danh sách sau khi sắp xếp** hoặc **tập tuple**, có lưu ý trường hợp hai khách trùng tên.

In [18]:
def run_183(customers, orders, query=sql183):
    db=sqlite3.connect(':memory:')
    try:
        db.executescript('CREATE TABLE Customers (id INTEGER PRIMARY KEY, name TEXT);'
                         'CREATE TABLE Orders (id INTEGER PRIMARY KEY, customerId INTEGER);')
        db.executemany('INSERT INTO Customers VALUES (?, ?)', customers)
        db.executemany('INSERT INTO Orders VALUES (?, ?)', orders)
        cur=db.execute(query)
        return [d[0] for d in cur.description],cur.fetchall()
    finally:
        db.close()

print(run_183(customers183,orders183))

(['Customers'], [('Henry',), ('Max',)])


## 22. Test 1 — bảng Orders rỗng

Nếu chưa có bất kỳ đơn hàng nào thì **mọi khách hàng đều chưa mua**. `LEFT JOIN` sẽ tạo một dòng cho mỗi khách với `o.id=NULL`, và `WHERE IS NULL` sẽ giữ tất cả.

Đây là phản ví dụ rất hay để phân biệt `LEFT JOIN` với `INNER JOIN`: INNER JOIN với Orders rỗng sẽ trả rỗng, tức **sai hoàn toàn** so với yêu cầu.

In [19]:
columns, rows = run_183([(1,'Joe'),(2,'Henry')], [])
assert columns==['Customers']
assert sorted(rows)==[('Henry',),('Joe',)]
print('PASS Orders rỗng:', rows)

PASS Orders rỗng: [('Joe',), ('Henry',)]


## 23. Test 2 — mọi khách đều có ít nhất một đơn

Nếu tất cả khách xuất hiện trong `Orders.customerId` ít nhất một lần, **không có khách nào cần xuất**.

Quan trọng: một khách có thể có **nhiều đơn**. Không phải cứ số đơn nhiều hơn số khách là chắc chắn ai cũng đã mua; phải nhìn **liên kết cụ thể theo ID**. Ở test này, cả Joe và Henry đều có đơn khớp.

In [20]:
columns, rows=run_183([(1,'Joe'),(2,'Henry')],[(10,1),(11,1),(12,2)])
assert columns==['Customers'] and rows==[]
print('PASS mọi khách đã mua → []')

PASS mọi khách đã mua → []


## 24. Test 3 — một khách có nhiều đơn, khách còn lại chưa mua

Nếu Sam có 3 đơn hàng, thì khi LEFT JOIN ta có thể có **3 dòng Sam** ở kết quả trung gian. Tuy nhiên tất cả các dòng đó đều có `o.id` khác `NULL`, nên **đều bị loại** khi lọc.

Henry chưa có đơn nào nên nhận **1 dòng với `o.id=NULL`** và được chọn. Kết quả chỉ có **1 dòng Henry** — không cần `DISTINCT` để loại các dòng Sam, vì Sam vốn không vượt qua `WHERE`.

Đây là cách chứng minh “có nhiều đơn không gây ra dòng trùng sai trong kết quả”.

In [21]:
columns, rows=run_183([(1,'Sam'),(2,'Henry')],[(1,1),(2,1),(3,1)])
assert rows==[('Henry',)]
print('PASS khách nhiều đơn:',rows)

PASS khách nhiều đơn: [('Henry',)]


## 25. Test 4 — tên trùng nhưng ID khác nhau

Đề đảm bảo `Customers.id` là duy nhất, nhưng **không nói `Customers.name` là duy nhất**. Có thể hai khách cùng tên `An` nhưng ID khác nhau; mỗi người là **một dòng khác nhau**.

Nếu cả hai chưa mua, câu SQL theo yêu cầu sẽ có **hai dòng tên An**. Tự ý thêm `DISTINCT` là nguy hiểm: nó **gộp hai khách thành một** dù bảng gốc có hai người.

**Nguyên tắc:** chỉ loại dòng trùng khi đề yêu cầu, không vì thấy hai chuỗi giống nhau mà xóa bớt.

In [22]:
columns, rows=run_183([(1,'An'),(2,'An'),(3,'Binh')],[(10,3)])
assert rows==[('An',),('An',)]
print('PASS tên trùng vẫn có hai bản ghi:', rows)

PASS tên trùng vẫn có hai bản ghi: [('An',), ('An',)]


## 26. Test 5 — khách duy nhất và bảng Customers rỗng

Hai trường hợp nhỏ nhất:

- Chỉ có **một** khách, chưa có đơn → trả một tên.
- **Không có khách** trong bảng Customers → không thể có khách cần xuất → trả bảng rỗng.

Dù tập dữ liệu đề gốc có thể không chứa tất cả tình huống này, chúng giúp kiểm chứng **logic** của lời giải. Truy vấn `LEFT JOIN` không phụ thuộc phải có ít nhất hai khách mới hoạt động.

In [23]:
assert run_183([(1,'Lan')],[])[1]==[('Lan',)]
assert run_183([],[])[1]==[]
print('PASS một khách và Customers rỗng')

PASS một khách và Customers rỗng


## 27. Tại sao `NOT EXISTS` cũng giải đúng? Bắt đầu từ câu hỏi “có tồn tại không?”

Thay vì ghép bảng, ta có thể nói: “Lấy khách `c` **nếu KHÔNG TỒN TẠI** đơn hàng có `customerId = c.id`”.

```sql
SELECT c.name AS Customers
FROM Customers c
WHERE NOT EXISTS (
    SELECT 1 FROM Orders o WHERE o.customerId = c.id
);
```

- `EXISTS (subquery)`: đúng nếu truy vấn con tìm được **ít nhất một dòng**.
- `NOT EXISTS`: phủ định, đúng khi truy vấn con **không có dòng nào**.
- `SELECT 1`: chọn giá trị 1 tượng trưng; EXISTS chỉ quan tâm **có dòng hay không**, không quan tâm giá trị cột.
- `o.customerId = c.id` liên hệ truy vấn con với khách của **dòng hiện tại** (correlated subquery).

Đây là cách rất sát câu tiếng Việt của đề.

In [24]:
sql183_exists = '''SELECT c.name AS Customers
FROM Customers c
WHERE NOT EXISTS (
    SELECT 1 FROM Orders o WHERE o.customerId = c.id
);'''
show_183(sql183_exists)
assert set(db183.execute(sql183_exists).fetchall())==set(db183.execute(sql183).fetchall())

Cột: ['Customers']
  ('Henry',)
  ('Max',)
Số dòng: 2


## 28. Trace `NOT EXISTS`: với mỗi khách ta hỏi một câu

Duyệt từng khách của Customers:

| Khách | Subquery `SELECT 1 ... WHERE o.customerId=c.id` có dòng? | `EXISTS` | `NOT EXISTS` | Kết quả |
|---|---|---|---|---|
| Joe (1) | Có đơn id=2 | True | False | bỏ |
| Henry (2) | Không có | False | True | chọn |
| Sam (3) | Có đơn id=1 | True | False | bỏ |
| Max (4) | Không có | False | True | chọn |

**Đừng nhầm `NOT EXISTS` với “không có bảng Orders”.** Bảng có thể rất nhiều đơn, nhưng *riêng khách này* vẫn chưa có đơn. Điều kiện tương quan `o.customerId=c.id` đảm bảo ta hỏi đúng từng khách.

In [25]:
for customer_id,name in customers183:
    matches=[oid for oid,cid in orders183 if cid==customer_id]
    print(name, '| subquery tìm thấy đơn', matches, '| NOT EXISTS =', not bool(matches))

Joe | subquery tìm thấy đơn [2] | NOT EXISTS = False
Henry | subquery tìm thấy đơn [] | NOT EXISTS = True
Sam | subquery tìm thấy đơn [1] | NOT EXISTS = False
Max | subquery tìm thấy đơn [] | NOT EXISTS = True


## 29. `NOT IN` nghe đơn giản, nhưng phải cẩn thận với `NULL`

Một cách viết khác có thể là:

```sql
SELECT name AS Customers
FROM Customers
WHERE id NOT IN (SELECT customerId FROM Orders);
```

**Khi mọi `Orders.customerId` là giá trị hợp lệ khác NULL**, cách này cho đáp án tương tự. Nhưng nếu danh sách con **chứa NULL**, `NOT IN` có thể trả kết quả bất ngờ vì logic ba trạng thái của SQL; một id không khớp số nào vẫn có thể bị đánh giá là **UNKNOWN**.

Bài chính được mô tả bằng khóa ngoại; ví dụ mở rộng tiếp theo **cố ý tạo dữ liệu `customerId=NULL` ngoài phần minh họa chuẩn** để thấy rủi ro chung khi tái sử dụng `NOT IN` trong bài khác. `NOT EXISTS` hoặc `LEFT JOIN ... IS NULL` thường dễ diễn giải hơn cho ý “không có bản ghi”.

In [26]:
sql183_not_in = '''SELECT name AS Customers FROM Customers
WHERE id NOT IN (SELECT customerId FROM Orders);'''
print('Với dữ liệu mẫu không có NULL:')
show_183(sql183_not_in)
print('Mở rộng chủ ý: Orders có một customerId=NULL:')
nullable_cust = [(1,'Joe'),(2,'Henry')]
nullable_order = [(10,1),(11,None)]
print('NOT IN:',run_183(nullable_cust,nullable_order,sql183_not_in)[1])
print('NOT EXISTS:',run_183(nullable_cust,nullable_order,sql183_exists)[1])
assert run_183(nullable_cust,nullable_order,sql183_exists)[1]==[('Henry',)]

Với dữ liệu mẫu không có NULL:
Cột: ['Customers']
  ('Henry',)
  ('Max',)
Số dòng: 2
Mở rộng chủ ý: Orders có một customerId=NULL:
NOT IN: []
NOT EXISTS: [('Henry',)]


## 30. Chứng minh lời giải `LEFT JOIN ... IS NULL` đúng

Xét một khách bất kỳ `c`:

1. Vì bảng trái là Customers, **`c` không bị mất đi** khi thực hiện LEFT JOIN.
2. **Nếu `c` có ít nhất một đơn**, một hay nhiều dòng Orders có `o.customerId = c.id` sẽ ghép vào; mọi dòng ghép này đều có `o.id` không NULL (khóa chính) → `WHERE o.id IS NULL` loại hết.
3. **Nếu `c` không có đơn**, LEFT JOIN sinh ra một dòng đại diện khách cùng các cột Orders là NULL → `WHERE o.id IS NULL` giữ lại đúng dòng đó.
4. `SELECT c.name AS Customers` đưa tên khách ra đúng cột cần nộp.

Do đó một khách được chọn **khi và chỉ khi chưa từng đặt hàng**. Lập luận này giải thích cả trường hợp nhiều đơn và khách không có đơn, không dựa vào vài test minh họa.

## 31. Chứng minh `NOT EXISTS` theo cùng cách hiểu

Với mỗi khách `c`, truy vấn con `SELECT 1 FROM Orders o WHERE o.customerId=c.id` tìm các đơn của **đúng khách này**.

- Có ít nhất một đơn → truy vấn con có dòng → `EXISTS` True → `NOT EXISTS` False → khách bị loại.
- Không có đơn → truy vấn con rỗng → `EXISTS` False → `NOT EXISTS` True → giữ khách.

Cả hai cách đều kiểm tra sự **vắng mặt của bản ghi liên quan**. Đây là pattern **anti join**, thường dùng cho “người chưa đăng ký”, “sản phẩm chưa bán”, “nhân viên chưa được phân công”, v.v.

## 32. Complexity — cần tách logic thuật toán và kế hoạch chạy SQL

Đặt `n` là **số khách** và `m` là **số đơn**.

- **Brute Force Python** hai vòng lặp: tối đa `O(n × m)` thời gian; `O(r)` bộ nhớ kết quả với `r` khách chưa mua, ngoài input.
- **Python dùng set các customerId đã đặt**: dự kiến `O(n+m)` thời gian trung bình và `O(m)` không gian phụ cho set (thực tế tối đa số ID khách khác nhau trong Orders).
- **SQL `LEFT JOIN` / `NOT EXISTS`**: thời gian vật lý phụ thuộc **chỉ mục trên `Orders.customerId`**, số dòng và query plan. Không có index có thể phải kiểm tra rất nhiều cặp; dùng hash join hoặc index lookups có thể giảm chi phí đáng kể. Không nên nói mọi LEFT JOIN luôn `O(n+m)`.
- Bộ nhớ do hệ quản trị SQL sử dụng để ghép bảng phụ thuộc cách thực thi, không phải mặc định O(1).

**Trả lời phỏng vấn tốt:** nêu ý tưởng anti join, giải thích lợi thế của index phù hợp, không hứa một Big-O tuyệt đối cho mọi database.

## 33. Mô phỏng tối ưu bằng Python `set`

Ta vừa thấy baseline cần quét Orders cho mỗi khách. Có thể cải tiến bằng cách:

1. Duyệt **một lần** qua Orders để tạo `ordered_ids = set(...)`.
2. Duyệt từng khách, kiểm tra `customer_id not in ordered_ids`.

`set` giữ các giá trị ID **khác nhau**; một người có 100 đơn vẫn chỉ có một ID trong set. Tra cứu `in` với set có thời gian **trung bình** gần O(1) (không phải luôn O(1) trong mọi tình huống).

Đây là minh họa sự khác nhau giữa tự xử lý dữ liệu bằng Python và để SQL quyết định kế hoạch thực thi. Code nộp #183 vẫn là SQL.

In [27]:
def never_order_set(customers,orders):
    ordered_ids = {customer_id for _, customer_id in orders}
    print('Tập khách đã đặt:',ordered_ids)
    result=[]
    for customer_id,name in customers:
        print(f'Đang xét {name}: id={customer_id}; thuộc set? {customer_id in ordered_ids}')
        if customer_id not in ordered_ids:
            result.append(name)
    return result

assert never_order_set(customers183,orders183)==['Henry','Max']

Tập khách đã đặt: {1, 3}
Đang xét Joe: id=1; thuộc set? True
Đang xét Henry: id=2; thuộc set? False
Đang xét Sam: id=3; thuộc set? True
Đang xét Max: id=4; thuộc set? False


## 34. Kiểm thử đối chiếu SQL và baseline Python trên nhiều mẫu

Một cách phát hiện lỗi: sinh các bộ dữ liệu nhỏ, chạy **hai cách độc lập** rồi so kết quả. Ta dùng `random.Random(183)` để mỗi lần chạy cùng một chuỗi dữ liệu.

Với các tên khác nhau, ta dùng `sorted` để bỏ phụ thuộc thứ tự SQL. Với trường hợp **tên trùng**, cần so danh sách (có tính số lần xuất hiện), không dùng `set` vì set sẽ vô tình xóa bớt khách có tên giống nhau.

Test giúp kiểm chứng nhiều tình huống, nhưng không thay thế chứng minh từng khách ở mục 30.

In [28]:
import random
rng183=random.Random(183)
for trial in range(100):
    n=rng183.randrange(0,12)
    customers=[(i+1, f'Ten_{i%4}') for i in range(n)]
    m=rng183.randrange(0,20) if n else 0
    orders=[(j+1,rng183.randint(1,n)) for j in range(m)]
    expected=sorted(never_order_baseline(customers,orders))
    actual=sorted(name for (name,) in run_183(customers,orders)[1])
    assert actual==expected,(trial,customers,orders,actual,expected)
print('PASS 100 ca ngẫu nhiên gồm cả khách trùng tên')

PASS 100 ca ngẫu nhiên gồm cả khách trùng tên


## 35. Các lỗi dễ mắc — đặt code sai cạnh cách sửa

| Lỗi | Vì sao sai? | Sửa thế nào? |
|---|---|---|
| `INNER JOIN` | Loại hết khách chưa có đơn trước khi lọc | `LEFT JOIN` |
| `ON c.id = o.id` | Ghép id khách với **id đơn**, sai quan hệ | `ON c.id = o.customerId` |
| `WHERE o.id = NULL` | So sánh NULL bằng `=` không trả True | `o.id IS NULL` |
| `WHERE o.id IS NOT NULL` | Chọn **khách đã mua** (ngược đề) | `IS NULL` |
| `SELECT o.id` | Đề yêu cầu **tên khách** | `SELECT c.name AS Customers` |
| `SELECT DISTINCT c.name` | Xóa mất khách riêng biệt nếu trùng tên | Không thêm DISTINCT tùy tiện |
| `NOT IN` có NULL | Logic ba trạng thái gây UNKNOWN | Dùng `NOT EXISTS`/`LEFT JOIN` |
| Bỏ `ON` đúng khóa | Tạo ghép sai hoặc tích Descartes | Kiểm tra c.id = o.customerId |

**Cách tự phát hiện:** chuẩn bị một khách chưa mua và một khách có đơn. Nếu chỉ thấy người mua, có thể bạn dùng sai JOIN hoặc điều kiện lọc.

## 36. Nhận dạng pattern “không tồn tại bản ghi liên quan”

Gặp những câu tiếng Việt như:

- Khách hàng **chưa từng** đặt hàng.
- Sinh viên **chưa** đăng ký lớp nào.
- Nhân viên **không có** công việc được giao.
- Sản phẩm **chưa xuất hiện** trong bảng đơn hàng.

Hãy hỏi **ba câu**:
1. Đối tượng nào phải được giữ đầy đủ? → bảng **bên trái**.
2. Dấu hiệu “có liên quan” nằm ở khóa nào? → `left.id = right.foreignKey`.
3. Bài muốn **có** hay **không có** liên quan? → `JOIN`/`EXISTS` vs `LEFT JOIN ... IS NULL`/`NOT EXISTS`.

Đây là một **tư duy tổng quát**, không chỉ một mẹo viết SQL cho #183.

## 37. Biến thể 1 — liệt kê khách *đã* từng đặt ít nhất một đơn

Nếu đảo yêu cầu thành “khách đã mua”, cách đơn giản là:

```sql
SELECT DISTINCT c.name
FROM Customers c
JOIN Orders o ON c.id=o.customerId;
```

**Nhưng lưu ý:** `DISTINCT c.name` có thể gộp **hai khách trùng tên**, nên nếu bài muốn mỗi **khách ID** một dòng, nên chọn **c.id, c.name** hoặc truy vấn EXISTS thay vì DISTINCT theo tên.

Biến thể chính xác hơn để giữ từng khách là:

```sql
SELECT c.id, c.name FROM Customers c
WHERE EXISTS (SELECT 1 FROM Orders o WHERE o.customerId = c.id);
```

Đây là **bài mở rộng**, không phải truy vấn nộp #183.

In [29]:
show_183('''SELECT c.id, c.name FROM Customers c
WHERE EXISTS(SELECT 1 FROM Orders o WHERE o.customerId=c.id) ORDER BY c.id;''')

Cột: ['id', 'name']
  (1, 'Joe')
  (3, 'Sam')
Số dòng: 2


[(1, 'Joe'), (3, 'Sam')]

## 38. Biến thể 2 — đếm số đơn của từng khách

Một yêu cầu hay gặp: “Mỗi khách có bao nhiêu đơn, kể cả người có 0 đơn?”. Ta có thể dùng `LEFT JOIN` và:

```sql
SELECT c.name, COUNT(o.id) AS order_count
FROM Customers c LEFT JOIN Orders o ON c.id=o.customerId
GROUP BY c.id, c.name;
```

**Vì sao dùng `COUNT(o.id)` thay vì `COUNT(*)`?**

- Khách chưa có đơn vẫn có **một dòng LEFT JOIN với o.id=NULL**.
- `COUNT(*)` sẽ đếm **dòng đại diện đó** thành 1, cho kết quả sai về số đơn.
- `COUNT(o.id)` chỉ đếm giá trị `o.id` không NULL → khách chưa mua có **0 đơn**.

Đây là cây cầu nối #183 với kiến thức `GROUP BY/COUNT` của #182.

In [30]:
count183 = '''SELECT c.id, c.name, COUNT(o.id) AS order_count
FROM Customers c LEFT JOIN Orders o ON c.id=o.customerId
GROUP BY c.id,c.name ORDER BY c.id;'''
rows=show_183(count183)
assert rows==[(1,'Joe',1),(2,'Henry',0),(3,'Sam',1),(4,'Max',0)]

Cột: ['id', 'name', 'order_count']
  (1, 'Joe', 1)
  (2, 'Henry', 0)
  (3, 'Sam', 1)
  (4, 'Max', 0)
Số dòng: 4


## 39. Vì sao không kiểm tra số đơn bằng `COUNT(*) = 0` sau LEFT JOIN?

Như biến thể vừa rồi: một khách chưa có đơn vẫn có một dòng **bên trái** được giữ lại sau LEFT JOIN, cho nên `COUNT(*)` theo nhóm khách đó là **1**, không phải 0.

Một truy vấn sai phổ biến:

```sql
SELECT c.name
FROM Customers c LEFT JOIN Orders o ON c.id=o.customerId
GROUP BY c.id,c.name
HAVING COUNT(*) = 0;
```

Nó **không** tìm ra khách nào chưa mua trong bảng trái có dòng. Sửa thành `HAVING COUNT(o.id) = 0`, hoặc đơn giản hơn dùng `WHERE o.id IS NULL` như lời giải chính.

In [31]:
print('Sai: HAVING COUNT(*)=0:')
show_183('''SELECT c.name FROM Customers c LEFT JOIN Orders o ON c.id=o.customerId
GROUP BY c.id,c.name HAVING COUNT(*)=0;''')
print('Đúng: HAVING COUNT(o.id)=0:')
show_183('''SELECT c.name FROM Customers c LEFT JOIN Orders o ON c.id=o.customerId
GROUP BY c.id,c.name HAVING COUNT(o.id)=0;''')

Sai: HAVING COUNT(*)=0:
Cột: ['name']
Số dòng: 0
Đúng: HAVING COUNT(o.id)=0:
Cột: ['name']
  ('Henry',)
  ('Max',)
Số dòng: 2


[('Henry',), ('Max',)]

## 40. Xem kế hoạch thực thi SQLite (chỉ để nhận biết)

`EXPLAIN QUERY PLAN` nói SQLite dự định đọc/tra cứu các bảng thế nào. Câu trả lời của SQL **không thay đổi** khi ta xem plan.

Trên dữ liệu nhỏ, SQLite có thể tự tạo cơ chế tra cứu tạm thời cho `Orders.customerId`. Các dòng của EXPLAIN không đại diện cho mọi MySQL server hoặc mọi tập dữ liệu lớn.

Mục đích ở đây là biết SQL **có thuật toán thực thi phía sau**, nên khi phân tích hiệu năng không nên khẳng định JOIN luôn là một vòng lặp cụ thể.

In [32]:
print('SQLite EXPLAIN QUERY PLAN:')
for row in db183.execute('EXPLAIN QUERY PLAN '+sql183):
    print(row)

SQLite EXPLAIN QUERY PLAN:
(3, 0, 0, 'SCAN c')
(7, 0, 0, 'BLOOM FILTER ON o (customerId=?)')
(16, 0, 0, 'SEARCH o USING AUTOMATIC COVERING INDEX (customerId=?) LEFT-JOIN')


# 41. TỰ LUYỆN — 6 bài tăng dần

**Đừng xem đáp án ngay.** Các bài tập được xếp: (1) dự đoán output, (2) lập bảng trace, (3) điền SQL, (4) tự viết cách khác, (5) biến thể dùng COUNT, (6) mini challenge.

Các code cell `TODO` được viết ở dạng định nghĩa hàm/biến để **không chạy lỗi khi dùng Run All**. Khi bạn hoàn thiện, hãy **tự bỏ dấu `#` ở các dòng assert** để thử. Đáp án nằm **sau tất cả bài tập**.

### Bài 1 — Dự đoán output từ hai bảng

Customers = `[(1,'An'), (2,'Binh'), (3,'Chi')]`; Orders = `[(10,1),(11,1)]`.

1. Ai là khách chưa từng đặt hàng?
2. An xuất hiện mấy dòng ở kết quả **LEFT JOIN trước lọc**?
3. Khi `WHERE o.id IS NULL`, An còn lại không?
4. Tên cột đầu ra LeetCode bắt buộc là gì?

**Trả lời bằng tay trước**, đừng nhìn code mẫu ở trên.

In [33]:
exercise1_customers183=[(1,'An'),(2,'Binh'),(3,'Chi')]
exercise1_orders183=[(10,1),(11,1)]
# Sau khi dự đoán, có thể chạy:
# print(run_183(exercise1_customers183,exercise1_orders183))

### Bài 2 — Trace `LEFT JOIN` từng dòng

Khách A (id=1), B (id=2), C (id=3), D (id=4); đơn có customerId lần lượt **1, 1, 4**.

Điền bảng:

| Khách | Số dòng sau LEFT JOIN | Có `o.id=NULL` không? | Chọn? |
|---|---:|---|---|
| A | ? | ? | ? |
| B | ? | ? | ? |
| C | ? | ? | ? |
| D | ? | ? | ? |

Nhớ: **khách chưa có đơn vẫn có một dòng** sau LEFT JOIN, nhưng cột đơn là NULL.

In [34]:
exercise2_trace183 = None  # TODO: tự điền {'A':(2,False,False), ...}

### Bài 3 — TODO: hoàn thiện câu SQL `LEFT JOIN ... IS NULL`

Hãy tự viết một chuỗi SQL đáp ứng **đúng format LeetCode**, tên cột **`Customers`**.

Trả lời trước 4 ý: bảng bên trái nào? JOIN loại gì? ghép theo khóa nào? dấu hiệu không có đơn là gì?

In [35]:
exercise3_sql183 = None
# TODO 1: SELECT c.name AS Customers
# TODO 2: FROM Customers c
# TODO 3: LEFT JOIN Orders o ON ...
# TODO 4: WHERE o.id IS NULL
# Sau khi tự làm:
# assert set(run_183(customers183,orders183,exercise3_sql183)[1])=={('Henry',),('Max',)}

### Bài 4 — Viết lại bằng `NOT EXISTS`

Không sử dụng JOIN; thay bằng câu hỏi cho từng khách: **không tồn tại** dòng Orders có customerId bằng id của khách.

Dùng `WHERE NOT EXISTS (SELECT 1 FROM Orders o WHERE ...)`. Nhớ: truy vấn con phải liên hệ với **khách đang xét**, không chỉ hỏi bảng Orders có rỗng hay không.

In [36]:
exercise4_sql183=None
# TODO: SELECT c.name AS Customers FROM Customers c WHERE NOT EXISTS (...)
# Tự chạy lại ví dụ có một khách đặt nhiều đơn; output vẫn loại đúng khách đó.

### Bài 5 — Biến thể: đếm số đơn của từng khách, kể cả 0

Viết truy vấn có 3 cột `id`, `name`, `order_count`, mỗi khách **một dòng**.

- Dùng LEFT JOIN.
- Dùng `COUNT(o.id)`, không dùng `COUNT(*)`.
- Dùng `GROUP BY c.id, c.name` để không gộp **hai người trùng tên**.

Đây là biến thể học thêm, **không phải output #183**.

In [37]:
exercise5_sql183=None
# TODO: SELECT c.id, c.name, COUNT(o.id) AS order_count
# FROM Customers c LEFT JOIN Orders o ON ...
# GROUP BY c.id, c.name

### Bài 6 — Mini challenge: tìm tên khách chưa đặt từ dữ liệu Python

Tự viết `exercise6_never_order183(customers, orders)` **không dùng SQL**:

- Tạo set các `customerId` có đơn.
- Duyệt từng `(id, name)` của Customers.
- Nếu id **không có** trong set, thêm `name` vào kết quả.
- Giữ nguyên số dòng nếu khách trùng tên.

Hãy tự so sánh với SQL trên dữ liệu: `[('An' id1), ('An' id2), ('Binh' id3)]`, chỉ Binh có đơn.

In [38]:
def exercise6_never_order183(customers,orders):
    # TODO 1: lấy tập ID khách từng đặt hàng từ Orders
    # TODO 2: tạo list result
    # TODO 3: duyệt customers, kiểm tra ID không ở tập
    # TODO 4: return result
    pass

# Sau khi làm xong, bật assert:
# assert exercise6_never_order183([(1,'An'),(2,'An'),(3,'Binh')],[(10,3)]) == ['An','An']

## 42. Gợi ý ba mức cho từng bài

| Bài | Cấp 1 — hướng tư duy | Cấp 2 — công cụ | Cấp 3 — gần đáp án |
|---|---|---|---|
| **1** | Một khách có nhiều đơn vẫn là người **đã mua** | Dùng customerId để kiểm tra | Binh, Chi chưa mua; An có hai dòng ghép |
| **2** | LEFT JOIN không xóa khách chưa có đơn | A có 2 đơn, D có 1; B, C không có | A:2 dòng, B:1 NULL, C:1 NULL, D:1 |
| **3** | Giữ mọi khách rồi loại ai đã ghép | `LEFT JOIN Orders o ON c.id=o.customerId` | `WHERE o.id IS NULL` |
| **4** | Với mỗi c, hỏi có đơn của c không | `NOT EXISTS (SELECT 1 ...)` | `... WHERE o.customerId=c.id` |
| **5** | Khách chưa mua vẫn có một dòng LEFT JOIN | `COUNT(o.id)` bỏ NULL | `GROUP BY c.id,c.name` |
| **6** | Đừng quét Orders từ đầu cho từng khách | Set các `customerId` | `if cid not in ordered_ids: result.append(name)` |

**Quy tắc đọc gợi ý:** chỉ đọc cấp 1 trước, tự thử thêm, rồi mới lên cấp 2/3. Đừng biến bài luyện thành đọc đáp án thụ động.

# 43. ĐÁP ÁN THAM KHẢO — Mở sau khi đã thử

## Đáp án Bài 1 và Bài 2: trace bằng tay

**Bài 1:** Binh và Chi chưa đặt; An có **hai dòng ghép** nhưng cả hai `o.id` đều khác NULL nên bị loại; cột nộp phải tên `Customers`.

**Bài 2:**

| Khách | Dòng LEFT JOIN | Có NULL? | Chọn? |
|---|---:|---|---|
| A (1) | 2 | Không | Không |
| B (2) | 1 | Có | Có |
| C (3) | 1 | Có | Có |
| D (4) | 1 | Không | Không |

**Vì sao B/C có 1 dòng?** LEFT JOIN vẫn giữ dòng khách không tìm được đối tác, với các cột đơn là `NULL`.

In [39]:
answer1_183=run_183(exercise1_customers183,exercise1_orders183)
assert sorted(answer1_183[1])==[('Binh',),('Chi',)]
print('Bài 1:',answer1_183)
print('Bài 2: A:2, B:1 NULL, C:1 NULL, D:1')

Bài 1: (['Customers'], [('Binh',), ('Chi',)])
Bài 2: A:2, B:1 NULL, C:1 NULL, D:1


### Đáp án Bài 3 — SQL `LEFT JOIN ... IS NULL`

Lời giải đúng phải giữ bảng **Customers bên trái**, ghép theo **ID khách trong Orders**, rồi lọc nơi **không có đơn**. `SELECT c.name AS Customers` đảm bảo cột nộp đúng tên.

In [40]:
answer3_sql183 = '''SELECT c.name AS Customers
FROM Customers c
LEFT JOIN Orders o ON c.id = o.customerId
WHERE o.id IS NULL;'''
assert sorted(run_183(customers183,orders183,answer3_sql183)[1])==[('Henry',),('Max',)]
print('PASS bài 3')

PASS bài 3


### Đáp án Bài 4 — `NOT EXISTS`

Truy vấn con là câu hỏi về **từng** dòng khách `c`, không phải câu hỏi “cả bảng Orders có rỗng không?”. `SELECT 1` chỉ biểu thị có dòng trong truy vấn con, không lấy một cột dữ liệu thực tế.

In [41]:
answer4_sql183 = '''SELECT c.name AS Customers FROM Customers c
WHERE NOT EXISTS (
    SELECT 1 FROM Orders o WHERE o.customerId = c.id
);'''
assert sorted(run_183(customers183,orders183,answer4_sql183)[1])==[('Henry',),('Max',)]
print('PASS bài 4')

PASS bài 4


### Đáp án Bài 5 — Đếm đúng số đơn (kể cả 0)

Dùng `COUNT(o.id)`: khi LEFT JOIN không tìm thấy đơn, `o.id` là NULL và không được đếm. Nếu dùng `COUNT(*)`, khách không có đơn lại được đếm thành 1 do có một dòng LEFT JOIN đại diện.

`GROUP BY c.id, c.name` cho **mỗi khách** một nhóm, tránh gộp hai khách có cùng tên.

In [42]:
answer5_sql183 = '''SELECT c.id, c.name, COUNT(o.id) AS order_count
FROM Customers c LEFT JOIN Orders o ON c.id=o.customerId
GROUP BY c.id,c.name;'''
columns, rows=run_183(customers183,orders183,answer5_sql183)
assert sorted(rows)==[(1,'Joe',1),(2,'Henry',0),(3,'Sam',1),(4,'Max',0)]
print('PASS bài 5:',rows)

PASS bài 5: [(1, 'Joe', 1), (2, 'Henry', 0), (3, 'Sam', 1), (4, 'Max', 0)]


### Đáp án Bài 6 — Python set, không làm mất khách trùng tên

Câu chuyện quan trọng: dù **cả hai khách cùng tên An**, nếu họ có hai ID khác nhau thì vẫn là **hai khách**. Ta kiểm tra **ID** trong `set` chứ không kiểm tra `name`. Kết quả có thể có hai chuỗi `'An'` hợp lệ và **không cần DISTINCT**.

In [43]:
def answer6_never_order183(customers,orders):
    ordered_ids={customer_id for _,customer_id in orders}
    result=[]
    for customer_id,name in customers:
        if customer_id not in ordered_ids:
            result.append(name)
    return result

assert answer6_never_order183([(1,'An'),(2,'An'),(3,'Binh')],[(10,3)]) == ['An','An']
assert answer6_never_order183(customers183,orders183)==['Henry','Max']
print('PASS bài 6')

PASS bài 6


## 44. Tổng kết: một khuôn mẫu để tự giải anti join

**Hãy tự hỏi:** “Tôi đang tìm các dòng *chưa có quan hệ* với bảng khác, hay các dòng *có quan hệ*?”

- Nếu tìm **chưa có**: giữ bảng gốc bằng `LEFT JOIN`, lọc `right.primary_key IS NULL`.
- Hoặc dùng `NOT EXISTS (SELECT 1 FROM right WHERE right.foreignKey = left.id)`.
- Ghép đúng **khóa liên quan**, không ghép hai cột `id` chỉ vì giống tên.

**SQL nộp #183:**

```sql
SELECT c.name AS Customers
FROM Customers c
LEFT JOIN Orders o ON c.id = o.customerId
WHERE o.id IS NULL;
```

**Checklist:** [ ] đúng bảng trái [ ] `ON c.id=o.customerId` [ ] `IS NULL`, không phải `= NULL` [ ] dùng `o.id` là khóa chính [ ] đầu ra `Customers` [ ] không thêm `DISTINCT` tùy tiện [ ] không dán Python lên LeetCode.

**Kỹ thuật chuyển giao:** phát hiện bản ghi **vắng mặt** là pattern phổ biến trong SQL phân tích dữ liệu, báo cáo và kiểm tra tính đầy đủ.